In [ ]:
from pathlib import Path
import sys

# Ensure project root is on sys.path for imports
project_root = Path.cwd()
while not (project_root / "main.py").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dataloader import load_illness_data

df = load_illness_data("SCZ", in_notebook=True)

Loading data for illness SCZ at /Users/leonackermann/Library/CloudStorage/GoogleDrive-leonmax.ackermann@googlemail.com/My Drive/Uni/Master/4/MasterThesis/ml-genetics4psychiatry/data/tmpDATA-Leon/donnees_MRI_SCZ_only_variants_clumping_p_thr_0.0001all.txt


In [19]:
df.head(5)

,Mean_intensity_3rd-Ventricle_whole-brain,Mean_intensity_4th-Ventricle_whole-brain,Mean_intensity_Brain-Stem_whole-brain,Mean_intensity_CSF_whole-brain,Mean_intensity_WM-hypointensities_whole-brain,Mean_intensity_Optic-Chiasm_whole-brain,Mean_intensity_CC-Posterior_whole-brain,Mean_intensity_CC-Mid-Posterior_whole-brain,Mean_intensity_CC-Central_whole-brain,Mean_intensity_CC-Mid-Anterior_whole-brain,...,Volume_S-postcentral_right,Volume_S-precentral-inf-part_right,Volume_S-precentral-sup-part_right,Volume_S-suborbital_right,Volume_S-subparietal_right,Volume_S-temporal-inf_right,Volume_S-temporal-sup_right,Volume_S-temporal-transverse_right,Z_scores_SCZ,ID
0,0.608651,0.417999,0.069745,0.118751,0.035988,-0.739803,1.208980,1.231560,0.780141,1.146150,...,-2.36435,-2.666830,0.707186,1.571280,-1.005540,-1.459920,-1.380530,-0.675617,3.980516,rs75878372
1,1.415920,1.123030,1.282310,1.587510,-0.347488,1.563280,2.182110,2.276960,1.213580,0.518312,...,-2.64181,-2.550490,-0.034571,0.555264,-0.749748,-3.499360,-3.379250,-1.665300,4.142574,rs150264563
2,-0.110527,-1.008280,-0.197095,-0.319888,0.126057,0.810311,-0.117957,-0.610847,-0.540175,-1.002590,...,-1.02813,0.272983,0.795672,-0.755772,0.135706,-0.654114,-0.153661,1.155150,3.974909,rs74911850
3,2.009210,1.407730,-0.855615,2.096700,-0.739741,-2.170680,0.553276,-0.059186,0.518272,0.902127,...,-1.09760,-1.469710,-2.289510,0.554313,1.490930,0.134280,0.404540,0.211052,3.928370,rs56086712
4,0.037307,-0.638911,-2.213110,-0.499057,-0.203933,-1.978070,1.718970,0.119012,-0.343785,0.134792,...,0.17855,-1.655620,-1.104240,0.955359,2.738340,0.903522,0.922400,1.136840,4.455699,rs17731


In [14]:
from dataloader import preprocess
import pandas as pd

X_train, y_train, X_test, y_test = preprocess(df=df, target="Z_scores_SCZ", testsize = 0.2)
# combine numpy arrays X_train and X_test into single array
X = pd.concat([pd.DataFrame(X_train), pd.DataFrame(X_test)], ignore_index=True)
y = pd.concat([pd.Series(y_train), pd.Series(y_test)], ignore_index=True)

# split X_train and y_train further into training and validation sets
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

In [10]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_squared_error,
    r2_score,
)
import xgboost as xgb
import matplotlib.pyplot as plt

np.random.seed(42)

In [11]:
xgb_regressor = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbosity=0,
)

xgb_regressor.fit(X_train, y_train)

y_pred = xgb_regressor.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

cv_scores_reg = cross_val_score(xgb_regressor, X, y, cv=5, scoring="r2")

In [15]:
# print results with the same formatting as above
print(f"Train MSE: {mean_squared_error(y_train, xgb_regressor.predict(X_train)):.4f}, Train R2: {r2_score(y_train, xgb_regressor.predict(X_train)):.4f}") 
print(f"Test MSE: {mse:.4f}, Test R2: {r2:.4f}")
print(f"Cross-Validation R2: {cv_scores_reg.mean():.4f} ± {cv_scores_reg.std():.4f}")

Train MSE: 0.2947, Train R2: 0.9866
Test MSE: 15.9568, Test R2: 0.2801
Cross-Validation R2: 0.3432 ± 0.0236


In [17]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_squared_error,
    r2_score,
)

param_grid = {  # Define parameter grid to test
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 6, 9],
    "learning_rate": [0.01, 0.1, 0.2],
    "subsample": [0.8, 0.9, 1.0],
}

grid_search = GridSearchCV(
    xgb.XGBRegressor(random_state=42, verbosity=0),
    param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_  # Evaluate on test set
test_accuracy = best_model.score(X_test, y_test)

In [18]:
# print results on results from box above
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Test R2 with Best Hyperparameters: {test_accuracy:.4f}")    


Best Hyperparameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.9}
Test R2 with Best Hyperparameters: 0.2894
